# Mimari Çeşitlilik v2 — ConvNeXt + Swin → 5-Model Ensemble

**Hedef:** TÜBİTAK 2209-A mel recall ≥ 0.85 hedefini ensemble çeşitliliği ile geçmek.

**Önceki ensemble:** 3-YOLO + multimodal = 0.8314 (hedeften −0.019)

**Bu notebook:** İki farklı mimari ailesi (CNN ailesi: ConvNeXt-Tiny + Transformer ailesi: Swin-Tiny) ekleyerek hata-dekorelasyonunu artırır. Beklenen mel recall katkısı +1.5-2.5%.

**Veri yapısı:**
```
/content/drive/MyDrive/Colab Notebooks/CiltKanseri/CiltKanseriProje/dataset_yolo/images/
├── train/   18,179 dosya (akiec_..., bcc_..., bkl_..., df_..., mel_..., nv_..., vasc_...)
├── val/      5,637 dosya
└── test/     3,684 dosya
```

**Dosya adı formatı:** `{class}_ISIC_xxxxxxx.jpg` veya `{class}_aug_NNN_ISIC_xxxxxxx.jpg` — etiket dosya adının ilk parçasından parse edilir.

**Çalıştırma:** Hücreleri sırayla çalıştır. **Cell 2'den sonra Runtime → Restart session yap, sonra cell 3'ten devam et.**

**Süreler (A100):**
- Veri kopyası: ~5-15 dk (Drive bağlantısına bağlı)
- ConvNeXt eğitimi: ~3-4 saat
- Swin eğitimi: ~4-5 saat
- Ensemble + threshold + TTA + ONNX: ~30 dk
- **Toplam: ~8-10 saat**


## 0. Ortam Kurulumu (Pillow + Paketler)

In [ ]:
# === ADIM 1/2: Pillow stabil sürüme zorla + diğer paketleri kur ===
!pip install -q --force-reinstall "Pillow==10.4.0"
!pip install -q -U timm onnx onnxconverter-common onnxruntime ultralytics
!pip install -q -U scikit-learn statsmodels seaborn matplotlib pandas

print('=' * 60)
print('PAKETLER KURULDU')
print('=' * 60)
print('Şimdi: Runtime menüsünden → Restart session (Ctrl+M nokta)')
print('Sonra: Cell 4 (sanity check) ile devam et.')
print('=' * 60)

## 1. Restart Sonrası Doğrulama

In [ ]:
# === ADIM 2/2 [RESTART SONRASI]: Ortam doğrulama ===
import torch, PIL, torchvision
print(f'PyTorch:     {torch.__version__}')
print(f'TorchVision: {torchvision.__version__}')
print(f'Pillow:      {PIL.__version__}  (10.4.0 olmalı)')
print(f'CUDA:        {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:         {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# PIL import testi (asıl hata buradaydı)
from PIL import Image, ImageDraw
print('PIL ImageDraw OK')

# Disk
import subprocess
df = subprocess.run(['df', '-h', '/content'], capture_output=True, text=True).stdout
print(f'\nDisk:\n{df}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Yapılandırma

In [ ]:
import os, time, json, gc, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# === Yollar ===
PROJECT_ROOT = '/content/drive/MyDrive/Colab Notebooks/CiltKanseri/CiltKanseriProje'
LOCAL_DATA   = '/content/dataset_yolo'                          # bu session'a kopyalanacak
YOLO_RUNS    = f'{PROJECT_ROOT}/yolo_runs'
DIVERSITY_DIR = f'{PROJECT_ROOT}/diversity_v2'                  # yeni outputs klasörü
PREDS_DIR    = f'{DIVERSITY_DIR}/predictions'

os.makedirs(DIVERSITY_DIR, exist_ok=True)
os.makedirs(PREDS_DIR, exist_ok=True)

# === Sınıflar (Ultralytics alfabetik) ===
CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
MEL_IDX = CLASS_TO_IDX['mel']

# === Hiperparametreler ===
IMG_SIZE = 384
BATCH_SIZE = 64
NUM_WORKERS = 4
EPOCHS = 25
PATIENCE = 7
LR = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'Yapılandırma OK')
print(f'  PROJECT_ROOT: {PROJECT_ROOT}')
print(f'  LOCAL_DATA:   {LOCAL_DATA}')
print(f'  DIVERSITY:    {DIVERSITY_DIR}')
print(f'  IMG_SIZE={IMG_SIZE}  BATCH={BATCH_SIZE}  EPOCHS={EPOCHS}  LR={LR}')

## 3. Veri Keşfi ve Lokal Kopya (Idempotent)

In [ ]:
# === Drive'daki dataset'i bul ve gerekirse lokal'e kopyala ===
DRIVE_DATASET = f'{PROJECT_ROOT}/dataset_yolo/images'

# Lokal kopya zaten var mı?
def check_local():
    if not os.path.isdir(f'{LOCAL_DATA}/images/train'):
        return None
    counts = {}
    for split in ('train', 'val', 'test'):
        d = f'{LOCAL_DATA}/images/{split}'
        if os.path.isdir(d):
            counts[split] = len([f for f in os.listdir(d) if f.endswith('.jpg')])
        else:
            counts[split] = 0
    return counts

local_counts = check_local()
if local_counts and local_counts.get('train', 0) > 17000:
    print(f'Lokal kopya MEVCUT — atlanıyor:')
    for s, n in local_counts.items():
        print(f'  {s}: {n:,}')
else:
    if not os.path.isdir(DRIVE_DATASET):
        print(f'HATA: {DRIVE_DATASET} bulunamadı.')
        print('Drive\'da dataset_yolo/images/ klasörünün var olduğundan emin ol.')
        raise FileNotFoundError(DRIVE_DATASET)
    print(f'Drive → Local kopya başlıyor:')
    print(f'  Kaynak: {DRIVE_DATASET}')
    print(f'  Hedef:  {LOCAL_DATA}/images')
    print(f'  Tahmini süre: 5-15 dk\n')
    os.makedirs(LOCAL_DATA, exist_ok=True)
    t0 = time.time()
    # rsync archive mode (-a = recursive + permissions); progress hidden for speed
    result = subprocess.run(['cp', '-r', DRIVE_DATASET, LOCAL_DATA + '/'],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(f'cp HATA: {result.stderr[:500]}')
        raise RuntimeError('Veri kopyası başarısız')
    print(f'Kopya bitti: {(time.time()-t0)/60:.1f} dk')
    local_counts = check_local()
    for s, n in local_counts.items():
        print(f'  {s}: {n:,}')

import subprocess
df_out = subprocess.run(['df', '-h', '/content'], capture_output=True, text=True).stdout
print(f'\nDisk durumu:\n{df_out}')

In [ ]:
# === Dosya adından etiket parse et, dataframe oluştur ===
# Format: {class}_ISIC_xxx.jpg veya {class}_aug_NNN_ISIC_xxx.jpg
import glob

def parse_class_from_filename(fname):
    """İlk underscore'a kadarki kısım sınıf adı."""
    name = os.path.basename(fname).lower()
    first_part = name.split('_', 1)[0]
    if first_part in CLASS_TO_IDX:
        return first_part
    # Bazı sınıflar 2 kelimeli olabilir (bizim için yok ama güvenlik için)
    return None

def build_df(split):
    paths = sorted(glob.glob(f'{LOCAL_DATA}/images/{split}/*.jpg'))
    rows = []
    bad = 0
    for p in paths:
        cls = parse_class_from_filename(p)
        if cls is None:
            bad += 1
            continue
        rows.append({'image_path': p, 'label': cls, 'y': CLASS_TO_IDX[cls]})
    df = pd.DataFrame(rows)
    if bad:
        print(f'  {split}: {bad} dosya parse edilemedi (sınıf tanınmadı)')
    return df

print('Dataframe inşa:')
train_df = build_df('train')
val_df   = build_df('val')
test_df  = build_df('test')
print(f'  train: {len(train_df):,}')
print(f'  val:   {len(val_df):,}')
print(f'  test:  {len(test_df):,}')

# Sınıf dağılımı
print('\nTrain sınıf dağılımı:')
print(train_df['label'].value_counts().sort_index().to_string())
print('\nTest sınıf dağılımı:')
print(test_df['label'].value_counts().sort_index().to_string())

# Class weights (mel için 1.5x boost)
class_counts = train_df['y'].value_counts().sort_index().values.astype(np.float32)
class_weights = (class_counts.mean() / class_counts)
class_weights[MEL_IDX] *= 1.5
class_weights_t = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print('\nClass weights (mel +50% boost):')
for c, w in zip(CLASS_NAMES, class_weights):
    marker = '  [MEL]' if c == 'mel' else ''
    print(f'  {c:6s}  w={w:.3f}{marker}')

## 4. Augmentation, Dataset ve Loaders

In [ ]:
import torchvision.transforms as T
from torchvision.transforms import InterpolationMode

train_tf = T.Compose([
    T.Resize(int(IMG_SIZE * 1.15), interpolation=InterpolationMode.BICUBIC),
    T.RandomCrop(IMG_SIZE),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomApply([T.ColorJitter(0.2, 0.2, 0.2, 0.05)], p=0.7),
    T.RandomApply([T.RandomRotation(30, interpolation=InterpolationMode.BICUBIC)], p=0.5),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.25, scale=(0.02, 0.20)),
])

eval_tf = T.Compose([
    T.Resize(int(IMG_SIZE * 1.15), interpolation=InterpolationMode.BICUBIC),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

tta_tf = T.Compose([
    T.Resize(int(IMG_SIZE * 1.15), interpolation=InterpolationMode.BICUBIC),
    T.CenterCrop(IMG_SIZE),
    T.RandomHorizontalFlip(p=1.0),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class ISICDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.tf = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(r['image_path']).convert('RGB')
        return self.tf(img), int(r['y']), i


def build_loaders():
    train_ds = ISICDataset(train_df, train_tf)
    val_ds   = ISICDataset(val_df,   eval_tf)
    test_ds  = ISICDataset(test_df,  eval_tf)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE*2, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = build_loaders()
print(f'Loaders OK — train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}')

# Sanity check
xb, yb, _ = next(iter(train_loader))
print(f'Batch test: x={tuple(xb.shape)}, y={tuple(yb.shape)}')
print(f'  x range: [{xb.min():.3f}, {xb.max():.3f}] (normalize sonrası)')
print(f'  y unique: {sorted(yb.unique().tolist())}')
assert xb.shape[1:] == (3, IMG_SIZE, IMG_SIZE), 'Tensor boyutu yanlış'
print('SANİTY OK')

## 5. Model Factory ve Eğitim Fonksiyonu

In [ ]:
import timm
from sklearn.metrics import (accuracy_score, f1_score, recall_score,
                              precision_score, balanced_accuracy_score, cohen_kappa_score)


def create_model(name, num_classes=NUM_CLASSES):
    """timm modelini yükle, son katmanı yeniden inşa et."""
    if 'swin' in name and 'window7_224' in name:
        # 224 native model'i 384'te interpolation
        m = timm.create_model(name, pretrained=True, num_classes=num_classes, img_size=IMG_SIZE)
    else:
        m = timm.create_model(name, pretrained=True, num_classes=num_classes)
    return m


def evaluate(model, loader):
    model.eval()
    ys, ps, probs = [], [], []
    with torch.no_grad(), autocast():
        for x, y, _ in loader:
            x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
            p = F.softmax(model(x), dim=1)
            probs.append(p.cpu().numpy())
            ys.append(y.cpu().numpy())
            ps.append(p.argmax(1).cpu().numpy())
    ys = np.concatenate(ys); ps = np.concatenate(ps); probs = np.concatenate(probs)
    m = {
        'accuracy':  accuracy_score(ys, ps),
        'macro_f1':  f1_score(ys, ps, average='macro', zero_division=0),
        'mel_recall':recall_score(ys, ps, labels=[MEL_IDX], average='macro', zero_division=0),
        'mel_prec':  precision_score(ys, ps, labels=[MEL_IDX], average='macro', zero_division=0),
        'bal_acc':   balanced_accuracy_score(ys, ps),
        'kappa':     cohen_kappa_score(ys, ps),
    }
    return m, ys, ps, probs


def train_model(model_name, tag, epochs=EPOCHS):
    print(f'\n{"=" * 70}')
    print(f'  {tag.upper()} ({model_name})')
    print(f'{"=" * 70}\n')

    model = create_model(model_name).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'Parametre: {n_params:.1f}M')

    criterion = nn.CrossEntropyLoss(weight=class_weights_t, label_smoothing=LABEL_SMOOTHING)
    optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        optim, max_lr=LR, epochs=epochs, steps_per_epoch=len(train_loader),
        pct_start=3/epochs, anneal_strategy='cos',
        div_factor=10.0, final_div_factor=100.0,
    )
    scaler = GradScaler()

    best_mel_recall = -1.0
    patience_left = PATIENCE
    save_path = f'{DIVERSITY_DIR}/{tag}_best.pt'
    history = []
    t0 = time.time()

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0; n_seen = 0
        for x, y, _ in train_loader:
            x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
            optim.zero_grad(set_to_none=True)
            with autocast():
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optim); scaler.update(); sched.step()
            train_loss += loss.item() * x.size(0); n_seen += x.size(0)
        train_loss /= n_seen

        m, _, _, _ = evaluate(model, val_loader)
        elapsed = (time.time() - t0) / 60.0
        history.append({'epoch': epoch+1, 'train_loss': train_loss, **m})

        msg = (f'Ep {epoch+1:2d}/{epochs} | loss={train_loss:.4f} | '
               f'val_acc={m["accuracy"]:.4f} mel_R={m["mel_recall"]:.4f} '
               f'mac_F1={m["macro_f1"]:.4f} | t={elapsed:.1f}min')

        if m['mel_recall'] > best_mel_recall:
            best_mel_recall = m['mel_recall']
            torch.save(model.state_dict(), save_path)
            patience_left = PATIENCE
            msg += '  [BEST]'
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(msg + '  [EARLY STOP]')
                break
        print(msg)

    print(f'\nBest val mel_recall: {best_mel_recall:.4f}')
    print(f'Best ckpt: {save_path}')

    # Test setinde tahmin
    model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    test_m, _, _, test_probs = evaluate(model, test_loader)
    _, _, _, val_probs = evaluate(model, val_loader)

    print(f'\nTest metrikleri ({tag}):')
    for k, v in test_m.items():
        print(f'  {k:12s}: {v:.4f}')

    np.save(f'{PREDS_DIR}/{tag}_test_probs.npy', test_probs)
    np.save(f'{PREDS_DIR}/{tag}_val_probs.npy',  val_probs)
    pd.DataFrame(history).to_csv(f'{DIVERSITY_DIR}/{tag}_history.csv', index=False)
    del model; gc.collect(); torch.cuda.empty_cache()
    return test_m, test_probs, val_probs


CONVNEXT_NAME = 'convnext_tiny.fb_in22k_ft_in1k_384'
SWIN_NAME     = 'swin_tiny_patch4_window7_224.ms_in22k_ft_in1k'
print('Model fonksiyonları hazır.')
print(f'ConvNeXt: {CONVNEXT_NAME}')
print(f'Swin:     {SWIN_NAME}')

## 6. ConvNeXt-Tiny Eğitimi (~3-4 saat A100)

In [ ]:
conv_test_m, conv_test_probs, conv_val_probs = train_model(CONVNEXT_NAME, 'convnext_tiny')

## 7. Swin-Tiny Eğitimi (~4-5 saat A100)

In [ ]:
swin_test_m, swin_test_probs, swin_val_probs = train_model(SWIN_NAME, 'swin_tiny')

## 8. YOLO Ailesinden Tahminleri Topla

In [ ]:
from ultralytics import YOLO

# Ultralytics class sırası = bizim CLASS_NAMES (alfabetik) — identity map
def yolo_predict(yolo_obj, df, name):
    paths = df['image_path'].tolist()
    probs = []
    for bs in range(0, len(paths), 64):
        batch = paths[bs:bs+64]
        results = yolo_obj.predict(batch, imgsz=IMG_SIZE, device=0, verbose=False)
        for r in results:
            probs.append(r.probs.data.cpu().numpy())
    probs = np.array(probs, dtype=np.float32)
    print(f'  {name}: {probs.shape}')
    return probs

yolo_paths = {
    'yolov8m':  f'{YOLO_RUNS}/yolov8m_cls/weights/best.pt',
    'yolo11m':  f'{YOLO_RUNS}/yolo11m_cls/weights/best.pt',
    'yolo26m':  f'{YOLO_RUNS}/yolo26m_cls/weights/best.pt',
}

yolo_test_probs = {}
yolo_val_probs  = {}

for name, pt in yolo_paths.items():
    if not os.path.exists(pt):
        print(f'  {name}: ATLANIYOR ({pt} yok)')
        continue
    print(f'\n{name}:')
    yolo = YOLO(pt)
    yolo_test_probs[name] = yolo_predict(yolo, test_df, f'{name} test')
    yolo_val_probs[name]  = yolo_predict(yolo, val_df,  f'{name} val')
    np.save(f'{PREDS_DIR}/{name}_test_probs.npy', yolo_test_probs[name])
    np.save(f'{PREDS_DIR}/{name}_val_probs.npy',  yolo_val_probs[name])
    del yolo; gc.collect(); torch.cuda.empty_cache()

print(f'\nToplanan YOLO modeli sayısı: {len(yolo_test_probs)}')

## 9. Bireysel Model Metrikleri

In [ ]:
y_test = test_df['y'].values
y_val  = val_df['y'].values

def quick(probs, y):
    pred = probs.argmax(1)
    return {
        'acc':   accuracy_score(y, pred),
        'macF1': f1_score(y, pred, average='macro', zero_division=0),
        'mel_R': recall_score(y, pred, labels=[MEL_IDX], average='macro', zero_division=0),
        'mel_P': precision_score(y, pred, labels=[MEL_IDX], average='macro', zero_division=0),
        'kappa': cohen_kappa_score(y, pred),
    }

# Tüm modellerin probs'larını topla
all_test = dict(yolo_test_probs)
all_test['convnext'] = conv_test_probs
all_test['swin']     = swin_test_probs

all_val = dict(yolo_val_probs)
all_val['convnext'] = conv_val_probs
all_val['swin']     = swin_val_probs

print(f'{"Model":<14}{"acc":>8}{"macF1":>8}{"mel_R":>8}{"mel_P":>8}{"kappa":>8}')
print('-' * 60)
for name, probs in all_test.items():
    m = quick(probs, y_test)
    print(f'{name:<14}{m["acc"]:>8.4f}{m["macF1"]:>8.4f}{m["mel_R"]:>8.4f}{m["mel_P"]:>8.4f}{m["kappa"]:>8.4f}')

## 10. Ensemble ve Threshold Tuning

In [ ]:
def ensemble(probs_dict):
    return np.mean(list(probs_dict.values()), axis=0)

# Ensemble varyantları
ensembles_test = {
    '3-YOLO':      ensemble({k: v for k, v in all_test.items() if k.startswith('yolo')}),
    'Diverse-5':   ensemble(all_test),  # 3 YOLO + ConvNeXt + Swin
}
ensembles_val = {
    '3-YOLO':      ensemble({k: v for k, v in all_val.items() if k.startswith('yolo')}),
    'Diverse-5':   ensemble(all_val),
}

print(f'{"Ensemble":<14}{"acc":>8}{"macF1":>8}{"mel_R":>8}{"mel_P":>8}{"kappa":>8}')
print('-' * 60)
for name, probs in ensembles_test.items():
    m = quick(probs, y_test)
    print(f'{name:<14}{m["acc"]:>8.4f}{m["macF1"]:>8.4f}{m["mel_R"]:>8.4f}{m["mel_P"]:>8.4f}{m["kappa"]:>8.4f}')

# === Threshold tuning (validation üzerinde) ===
def threshold_metrics(probs, y, theta):
    pred = probs.argmax(1).copy()
    mel_p = probs[:, MEL_IDX]
    flip = (mel_p >= theta) & (pred != MEL_IDX)
    pred[flip] = MEL_IDX
    return {
        'theta': theta,
        'acc':    accuracy_score(y, pred),
        'macF1':  f1_score(y, pred, average='macro', zero_division=0),
        'mel_R':  recall_score(y, pred, labels=[MEL_IDX], average='macro', zero_division=0),
        'mel_P':  precision_score(y, pred, labels=[MEL_IDX], average='macro', zero_division=0),
        'kappa':  cohen_kappa_score(y, pred),
    }

print('\n' + '=' * 70)
print('THRESHOLD TUNING (Diverse-5 ensemble, validation üzerinde)')
print('=' * 70)
thresholds = np.arange(0.10, 0.45, 0.02)
rows_val = [threshold_metrics(ensembles_val['Diverse-5'], y_val, t) for t in thresholds]
df_val = pd.DataFrame(rows_val)

# Mel recall ≥ 0.85'i sağlayan eşikler arasında max precision'lı olanı seç
ok = df_val[df_val['mel_R'] >= 0.85].copy()
if len(ok) == 0:
    chosen = df_val.loc[df_val['mel_R'].idxmax()]
    reason = 'mel_R maks (0.85 sağlanmadı)'
else:
    chosen = ok.loc[ok['mel_P'].idxmax()]
    reason = 'mel_R ≥ 0.85 + max precision'

theta_star = float(chosen['theta'])
print(f'Seçilen θ_mel = {theta_star:.3f}  [{reason}]')
print(f'Val @ θ={theta_star:.3f}:')
for k in ('acc', 'macF1', 'mel_R', 'mel_P', 'kappa'):
    print(f'  {k:8s}: {chosen[k]:.4f}')

# Test setinde uygula
test_result = threshold_metrics(ensembles_test['Diverse-5'], y_test, theta_star)
print(f'\nTest @ θ={theta_star:.3f}:')
for k in ('acc', 'macF1', 'mel_R', 'mel_P', 'kappa'):
    print(f'  {k:8s}: {test_result[k]:.4f}')

df_val.to_csv(f'{DIVERSITY_DIR}/threshold_scan.csv', index=False)
print(f'\nKaydedildi: threshold_scan.csv')

## 11. McNemar Testleri (Ensemble Çeşitliliğinin İstatistiksel Kanıtı)

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
from itertools import combinations

def mcnemar_pair(probs_a, probs_b, y):
    pa = probs_a.argmax(1); pb = probs_b.argmax(1)
    a_ok = (pa == y); b_ok = (pb == y)
    b01 = int(np.sum( a_ok & ~b_ok))
    b10 = int(np.sum(~a_ok &  b_ok))
    exact = (b01 + b10) < 25
    r = mcnemar([[0, b01], [b10, 0]], exact=exact, correction=True)
    return {'b01': b01, 'b10': b10, 'stat': float(r.statistic), 'p': float(r.pvalue)}

pairs = list(combinations(all_test.keys(), 2))
alpha_corr = 0.05 / len(pairs)
print(f'Toplam ikili karşılaştırma: {len(pairs)}, Bonferroni α = {alpha_corr:.4f}\n')
print(f'{"A":<14} vs {"B":<14}  {"b01":>5} {"b10":>5}  {"χ²":>7}  {"p":>8}  {"sig":>5}')
print('-' * 70)
rows = []
for a, b in pairs:
    r = mcnemar_pair(all_test[a], all_test[b], y_test)
    sig = 'Evet' if r['p'] < alpha_corr else 'Hayır'
    rows.append({'A': a, 'B': b, **r, 'significant': sig})
    print(f'{a:<14} vs {b:<14}  {r["b01"]:>5} {r["b10"]:>5}  {r["stat"]:>7.2f}  {r["p"]:>8.4f}  {sig:>5}')

pd.DataFrame(rows).to_csv(f'{DIVERSITY_DIR}/mcnemar.csv', index=False)

## 12. TTA + Final TÜBİTAK Tablosu

In [ ]:
# === TTA: ConvNeXt ve Swin için horizontal flip averaging ===
def tta_predict(model_name, tag):
    model = create_model(model_name).to(DEVICE)
    model.load_state_dict(torch.load(f'{DIVERSITY_DIR}/{tag}_best.pt', map_location=DEVICE))
    model.eval()
    test_ds_orig = ISICDataset(test_df, eval_tf)
    test_ds_flip = ISICDataset(test_df, tta_tf)
    lo = DataLoader(test_ds_orig, batch_size=BATCH_SIZE*2, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True)
    lf = DataLoader(test_ds_flip, batch_size=BATCH_SIZE*2, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True)
    def collect(loader):
        out = []
        with torch.no_grad(), autocast():
            for x, _, _ in loader:
                x = x.to(DEVICE, non_blocking=True)
                out.append(F.softmax(model(x), dim=1).cpu().numpy())
        return np.concatenate(out)
    p1 = collect(lo); p2 = collect(lf)
    del model; gc.collect(); torch.cuda.empty_cache()
    return (p1 + p2) / 2.0

print('ConvNeXt TTA...')
conv_tta = tta_predict(CONVNEXT_NAME, 'convnext_tiny')
print('Swin TTA...')
swin_tta = tta_predict(SWIN_NAME, 'swin_tiny')
np.save(f'{PREDS_DIR}/convnext_tiny_test_tta.npy', conv_tta)
np.save(f'{PREDS_DIR}/swin_tiny_test_tta.npy', swin_tta)

# TTA-Diverse-5 ensemble
all_test_tta = dict(yolo_test_probs)
all_test_tta['convnext'] = conv_tta
all_test_tta['swin']     = swin_tta
ens_tta_test = ensemble(all_test_tta)

# Final tablo
print('\n' + '=' * 80)
print('FİNAL KONFİGÜRASYONLAR')
print('=' * 80)
configs = [
    ('YOLO11 tek',              all_test['yolo11m'], 0.0),
    ('3-YOLO ensemble',         ensembles_test['3-YOLO'], 0.0),
    ('Diverse-5 ensemble',      ensembles_test['Diverse-5'], 0.0),
    ('Diverse-5 + TTA',         ens_tta_test, 0.0),
    (f'Diverse-5 @ θ={theta_star:.2f}',    ensembles_test['Diverse-5'], theta_star),
    (f'Diverse-5+TTA @ θ={theta_star:.2f}', ens_tta_test, theta_star),
]
results = []
print(f'{"Konfigürasyon":<32}{"acc":>8}{"macF1":>8}{"mel_R":>8}{"mel_P":>8}{"kappa":>8}')
print('-' * 80)
for name, probs, theta in configs:
    if theta > 0:
        m = threshold_metrics(probs, y_test, theta)
        m['acc'] = m['acc']; m['macF1'] = m['macF1']; m['mel_R'] = m['mel_R']
    else:
        m = quick(probs, y_test)
    results.append({'config': name, **m})
    print(f'{name:<32}{m["acc"]:>8.4f}{m["macF1"]:>8.4f}{m["mel_R"]:>8.4f}{m["mel_P"]:>8.4f}{m["kappa"]:>8.4f}')

df_final = pd.DataFrame(results)
df_final.to_csv(f'{DIVERSITY_DIR}/final.csv', index=False)

# TÜBİTAK karşılaştırma
TUBITAK = {'acc': 0.90, 'macF1': 0.85, 'mel_R': 0.85, 'kappa': 0.85}
best_row = df_final.iloc[df_final['mel_R'].idxmax()]
print(f'\nEn iyi mel_R konfigürasyonu: {best_row["config"]}')
print('\nTÜBİTAK Karşılaştırma:')
print(f'  {"Metrik":<10}{"Mevcut":>10}{"Hedef":>10}{"Δ":>10}{"Durum":>15}')
print('  ' + '-' * 55)
for k in ('acc', 'macF1', 'mel_R', 'kappa'):
    val = float(best_row[k]); tgt = TUBITAK[k]; delta = val - tgt
    status = 'Karşılandı' if val >= tgt else 'Açık'
    print(f'  {k:<10}{val:>10.4f}{tgt:>10.4f}{delta:>+10.4f}{status:>15}')

## 13. ONNX FP16 Export (Mobil için)

In [ ]:
import onnx
from onnxconverter_common import float16

def export_onnx_fp16(model_name, tag, out_path):
    m = create_model(model_name).to(DEVICE)
    m.load_state_dict(torch.load(f'{DIVERSITY_DIR}/{tag}_best.pt', map_location=DEVICE))
    m.eval()
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    tmp = out_path.replace('.onnx', '_fp32.onnx')
    torch.onnx.export(m, dummy, tmp,
                      input_names=['images'], output_names=['logits'],
                      dynamic_axes={'images': {0: 'batch'}, 'logits': {0: 'batch'}},
                      opset_version=14, do_constant_folding=True)
    model_fp32 = onnx.load(tmp)
    model_fp16 = float16.convert_float_to_float16(model_fp32, keep_io_types=True)
    onnx.save(model_fp16, out_path)
    size_mb = os.path.getsize(out_path) / (1024**2)
    os.remove(tmp)
    del m; gc.collect(); torch.cuda.empty_cache()
    return size_mb

print('ONNX FP16 export:')
conv_size = export_onnx_fp16(CONVNEXT_NAME, 'convnext_tiny', f'{DIVERSITY_DIR}/convnext_tiny_fp16.onnx')
swin_size = export_onnx_fp16(SWIN_NAME,     'swin_tiny',     f'{DIVERSITY_DIR}/swin_tiny_fp16.onnx')
print(f'  ConvNeXt-Tiny FP16: {conv_size:.2f} MB   (≤25 MB hedef: ' + ('Karşılandı' if conv_size <= 25 else 'Aşıldı') + ')')
print(f'  Swin-Tiny FP16:     {swin_size:.2f} MB   (≤25 MB hedef: ' + ('Karşılandı' if swin_size <= 25 else 'Aşıldı') + ')')

## 14. Çıktılar Özeti

**Üretilen dosyalar (`{PROJECT_ROOT}/diversity_v2/` altında):**
- `convnext_tiny_best.pt`, `swin_tiny_best.pt` — eğitilmiş checkpoint'ler
- `convnext_tiny_history.csv`, `swin_tiny_history.csv` — eğitim eğrileri
- `threshold_scan.csv` — validation threshold-metric eğrileri
- `mcnemar.csv` — tüm ikili McNemar test sonuçları
- `final.csv` — final konfigürasyon karşılaştırma tablosu
- `convnext_tiny_fp16.onnx`, `swin_tiny_fp16.onnx` — mobil için

**Tahminler (`predictions/`):**
- Her modelin val/test softmax probs (.npy)
- TTA tahminleri

**Tezde güncellenecek yerler:**
1. Bölüm 4 — Tablo 4.9 (TÜBİTAK karşılaştırma): yeni satır "Diverse-5 + TTA @ θ_mel"
2. Bölüm 4 — McNemar 15-pair tablosu eklenir
3. Bölüm 4 — Yeni alt-bölüm: "Mimari Çeşitlilik ile Ensemble"
4. Bölüm 5 — Sonuç güncellenir (mel recall ≥ 0.85 sağlanırsa H2 kısmen karşılandı yazılır)

**Eğer mel recall hâlâ < 0.85:**
- Threshold daha aşağı (0.10-0.15) zorla; precision düşse de TÜBİTAK için recall öncelikli.
- Class weight mel boost 1.5x → 2.0x yap, yeniden eğit.
- TTA'ya vertical flip + multi-scale ekle.
